In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.56.0", "tokenizers>=0.21", "datasets",
    "scikit-learn", "matplotlib", "tqdm", "huggingface_hub"])
print("installed")


In [ ]:
import os, math, time
import numpy as np
import torch
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, classification_report)
from tqdm.auto import tqdm
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
import matplotlib.pyplot as plt

import huggingface_hub
if os.getenv("HF_TOKEN"):
    huggingface_hub.login(os.environ["HF_TOKEN"])

import warnings
warnings.filterwarnings("ignore")
os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

N_GPUS = torch.cuda.device_count()
DTYPE = torch.bfloat16 if N_GPUS else torch.float32
DEVICE = "cuda" if N_GPUS else "cpu"
for i in range(N_GPUS):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
print(f"GPUs: {N_GPUS}, dtype: {DTYPE}")


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class BPETokenizer:
    def __init__(self, tokenizer, max_len, pad_id=0):
        self.tok = tokenizer
        self.max_len = max_len
        self.pad_id = pad_id
        self.word2idx = tokenizer.get_vocab()
        tokenizer.enable_padding(length=max_len, pad_id=pad_id, pad_token="<PAD>")
        tokenizer.enable_truncation(max_length=max_len)

    def encode(self, text):
        return self.tok.encode(text).ids


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.0):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.scale = math.sqrt(self.d_k)

    def forward(self, q, k, v, mask=None):
        b = q.size(0)
        Q = self.W_q(q).view(b, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(k).view(b, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(v).view(b, -1, self.num_heads, self.d_k).transpose(1, 2)
        scores = (Q @ K.transpose(-2, -1)) / self.scale
        if mask is not None:
            scores = scores.masked_fill(mask == 1, float("-inf"))
        attn = self.dropout(F.softmax(scores, dim=-1))
        ctx = (attn @ V).transpose(1, 2).contiguous().view(b, -1, self.d_model)
        return self.W_o(ctx), attn


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.ReLU()

    def forward(self, x):
        return self.linear2(self.dropout(self.activation(self.linear1(x))))


class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        a, _ = self.attention(x, x, x, mask)
        x = self.norm1(x + a)
        x = self.norm2(x + self.ffn(x))
        return x


class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=512, num_heads=8, num_layers=6,
                 d_ff=2048, max_len=512, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.encoder_blocks = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, 1)

    def forward(self, x, mask=None):
        x = self.dropout(self.embedding(x))
        x = self.pos_enc(x)
        for blk in self.encoder_blocks:
            x = blk(x, mask)
        x = x.mean(dim=1)
        return self.classifier(x).squeeze(-1)


print("architecture ready")


In [ ]:
MAX_SEQ_LEN   = 320
BATCH_SIZE    = 64
LR            = 2e-4
WEIGHT_DECAY  = 1e-2
DROPOUT       = 0.1
NUM_EPOCHS    = 8
EARLY_STOP    = 3
D_MODEL       = 512
NUM_HEADS     = 8
NUM_LAYERS    = 4
D_FF          = 2048
BPE_VOCAB     = 12000
TIME_BUDGET_S = 4 * 3600


In [ ]:
def prep(df, keep_meta=False):
    df = df.dropna(subset=["response", "response_harm_label"])
    df = df[df["response_harm_label"].isin(["harmful", "unharmful"])].copy()
    df["text"]  = df["response"].astype(str).str.strip()
    df["label"] = (df["response_harm_label"] == "harmful").astype(int)
    df = df[df["text"].str.len() > 10].reset_index(drop=True)
    cols = ["text", "label"]
    if keep_meta:
        cols += ["adversarial", "subcategory", "prompt_harm_label",
                "response_refusal_label"]
    return df[cols]

train_df = prep(load_dataset("allenai/wildguardmix",
                            "wildguardtrain", split="train").to_pandas())
test_df  = prep(load_dataset("allenai/wildguardmix",
                            "wildguardtest", split="test").to_pandas(),
                 keep_meta=True)
train_df = train_df.drop_duplicates(subset=["text", "label"]).reset_index(drop=True)
print(f"train: {len(train_df)}  test: {len(test_df)}")


In [ ]:
bpe = Tokenizer(BPE(unk_token="<UNK>"))
bpe.pre_tokenizer = Whitespace()
trainer = BpeTrainer(vocab_size=BPE_VOCAB,
                     special_tokens=["<PAD>", "<UNK>"],
                     min_frequency=2)
bpe.train_from_iterator(train_df["text"].tolist(), trainer)
clf_tokenizer = BPETokenizer(bpe, max_len=MAX_SEQ_LEN, pad_id=0)
VOCAB_SIZE = len(clf_tokenizer.word2idx)
print(f"vocab: {VOCAB_SIZE}")


In [ ]:
import torch.utils.data as tud

class TextDS(tud.Dataset):
    def __init__(self, texts, labels, tok):
        self.texts, self.labels, self.tok = texts, labels, tok
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        ids = self.tok.encode(self.texts[i])
        return (torch.tensor(ids, dtype=torch.long),
                torch.tensor(self.labels[i], dtype=torch.float))

tr_txt, va_txt, tr_lbl, va_lbl = train_test_split(
    train_df["text"].tolist(), train_df["label"].tolist(),
    test_size=0.15, random_state=42, stratify=train_df["label"].tolist())
tr_dl = tud.DataLoader(TextDS(tr_txt, tr_lbl, clf_tokenizer),
                       batch_size=BATCH_SIZE, shuffle=True)
va_dl = tud.DataLoader(TextDS(va_txt, va_lbl, clf_tokenizer),
                       batch_size=BATCH_SIZE)
test_dl = tud.DataLoader(TextDS(test_df["text"].tolist(),
                                test_df["label"].tolist(), clf_tokenizer),
                          batch_size=BATCH_SIZE)


In [ ]:
model = TransformerClassifier(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL, num_heads=NUM_HEADS, num_layers=NUM_LAYERS,
    d_ff=D_FF, max_len=MAX_SEQ_LEN, dropout=DROPOUT,
).to(DEVICE)
print(f"params: {sum(p.numel() for p in model.parameters()):,}")

n_pos = sum(tr_lbl);  n_neg = len(tr_lbl) - n_pos
pos_weight = torch.tensor([n_neg / n_pos], device=DEVICE)
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR,
                               weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", patience=2, factor=0.5)


In [ ]:
history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_f1": []}
best_f1, best_state, patience = 0.0, None, EARLY_STOP
start = time.time()

for ep in tqdm(range(NUM_EPOCHS), desc="epochs"):
    if time.time() - start >= TIME_BUDGET_S:
        print("time budget hit")
        break

    model.train()
    tr_losses = []
    for x, y in tr_dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        mask = (x == 0).unsqueeze(1).unsqueeze(2)
        loss = criterion(model(x, mask), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tr_losses.append(loss.item())

    model.eval()
    va_losses, probs_all, labels_all = [], [], []
    with torch.no_grad():
        for x, y in va_dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            mask = (x == 0).unsqueeze(1).unsqueeze(2)
            logits = model(x, mask)
            va_losses.append(criterion(logits, y).item())
            probs_all.append(torch.sigmoid(logits).cpu().numpy())
            labels_all.append(y.cpu().numpy())

    probs_arr  = np.concatenate(probs_all)
    labels_arr = np.concatenate(labels_all)
    preds_arr  = (probs_arr >= 0.5).astype(int)
    va_acc = accuracy_score(labels_arr, preds_arr)
    va_f1  = f1_score(labels_arr, preds_arr, zero_division=0)
    tr_l   = float(np.mean(tr_losses))
    va_l   = float(np.mean(va_losses))
    scheduler.step(va_f1)
    history["train_loss"].append(tr_l)
    history["val_loss"].append(va_l)
    history["val_acc"].append(va_acc)
    history["val_f1"].append(va_f1)
    print(f"ep {ep+1:02d} | tr={tr_l:.4f} va={va_l:.4f} "
          f"acc={va_acc:.4f} f1={va_f1:.4f}")

    if va_f1 > best_f1:
        best_f1  = va_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience = EARLY_STOP
    else:
        patience -= 1
        if patience <= 0:
            print(f"early stop at ep {ep+1}")
            break

if best_state is not None:
    model.load_state_dict(best_state)
print(f"best val_f1: {best_f1:.4f}")


In [ ]:
model.eval()
all_probs, all_labels = [], []
with torch.no_grad():
    for x, y in test_dl:
        x = x.to(DEVICE)
        mask = (x == 0).unsqueeze(1).unsqueeze(2)
        all_probs.append(torch.sigmoid(model(x, mask)).cpu().numpy())
        all_labels.append(y.numpy())
probs  = np.concatenate(all_probs)
labels = np.concatenate(all_labels)
preds  = (probs >= 0.5).astype(int)
print(f"acc={accuracy_score(labels, preds):.4f}  "
      f"f1={f1_score(labels, preds, zero_division=0):.4f}  "
      f"roc-auc={roc_auc_score(labels, probs):.4f}  "
      f"pr-auc={average_precision_score(labels, probs):.4f}")
print(confusion_matrix(labels, preds))
print(classification_report(labels, preds, target_names=["unharmful", "harmful"],
                            zero_division=0))


In [ ]:
import zipfile
CKPT   = "transformer_classifier.pt"
TOKOUT = "bpe_tokenizer.json"
cfg = {"vocab_size": VOCAB_SIZE, "d_model": D_MODEL, "num_heads": NUM_HEADS,
       "num_layers": NUM_LAYERS, "d_ff": D_FF, "max_len": MAX_SEQ_LEN,
       "dropout": DROPOUT}
torch.save({"config": cfg, "model_state": model.state_dict()}, CKPT)
bpe.save(TOKOUT)

ZIP = "classifier_bundle.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(CKPT);  zf.write(TOKOUT)
print(f"saved {CKPT} + {TOKOUT}  ->  {ZIP} "
      f"({os.path.getsize(ZIP)/1e6:.1f} MB)")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"],   label="val")
axes[0].set_title("loss"); axes[0].set_xlabel("epoch")
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(history["val_acc"], label="acc")
axes[1].plot(history["val_f1"],  label="f1")
axes[1].set_title("val metrics"); axes[1].set_xlabel("epoch")
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("classifier_training.png", dpi=120)
plt.show()
